# Smoke-training и overfit-test

- Одна короткая эпоха
- Backpropagation для model и loss
- Переобучение на фиксированном batch


In [ ]:
from itertools import chain
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from mden_battery.data import (
    PreparedWindowIterableDataset,
    WindowConfig,
)
from mden_battery.loss import MDENJointLoss
from mden_battery.model import MDEN, MDENConfig
from mden_battery.training import (
    overfit_one_batch,
    seed_everything,
    train_epoch,
)


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "prepared_log_age"
INDEX_PATH = DATA_ROOT / "prepared_index.csv"
SCALER_PATH = DATA_ROOT / "scaler.csv"

SEED = 42
BATCH_SIZE = 16
SMOKE_BATCHES = 4
OVERFIT_BATCH_SIZE = 4
OVERFIT_STEPS = 1_000
OVERFIT_LR = 3e-3
OVERFIT_TARGET_RMSE = 0.02
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

WINDOW_CONFIG = WindowConfig(
    input_length=32,
    horizon=8,
    stride=8,
)
SMOKE_MODEL_CONFIG = MDENConfig(
    input_dim=5,
    horizon=8,
    top_k=1,
    fem_hidden_channels=8,
    fem_groups=2,
    fem_dropout=0.0,
    soc_depth=1,
    shared_depth=1,
    soh_depth=1,
    conv_rounds=1,
    fusion_hidden=8,
    sequence_dim=16,
    mamba_d_state=4,
    mamba_d_conv=3,
    mamba_expand=1,
    mamba_backend="native",
    prediction_hidden=32,
    dropout=0.0,
)


In [ ]:
def build_loader(split: str, batch_size: int) -> DataLoader:
    """Build a streaming loader for the smoke test."""
    dataset = PreparedWindowIterableDataset(
        INDEX_PATH,
        split=split,
        config=WINDOW_CONFIG,
        scaler_csv=SCALER_PATH,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    )


In [ ]:
def slice_batch(
    batch: dict[str, torch.Tensor],
    size: int,
) -> dict[str, torch.Tensor]:
    """Take a fixed non-empty subset of a batch."""
    selected = {
        name: tensor[:size].clone()
        for name, tensor in batch.items()
    }
    if selected["x"].shape[0] == 0:
        raise ValueError("Cannot overfit an empty batch")
    return selected


## Одна smoke-эпоха


In [ ]:
assert INDEX_PATH.is_file(), INDEX_PATH
assert SCALER_PATH.is_file(), SCALER_PATH

seed_everything(SEED)
train_loader = build_loader("train", BATCH_SIZE)
model = MDEN(SMOKE_MODEL_CONFIG).to(DEVICE)
criterion = MDENJointLoss(
    feature_dim=SMOKE_MODEL_CONFIG.input_dim
).to(DEVICE)
optimizer = torch.optim.AdamW(
    chain(model.parameters(), criterion.parameters()),
    lr=1e-3,
    weight_decay=1e-4,
)

smoke_metrics = train_epoch(
    model,
    criterion,
    train_loader,
    optimizer,
    DEVICE,
    gradient_clip_norm=1.0,
    max_batches=SMOKE_BATCHES,
)
assert all(
    torch.isfinite(torch.tensor(value))
    for value in smoke_metrics.values()
)
print(smoke_metrics)


## Overfit-test


In [ ]:
seed_everything(SEED)
fixed_batch = slice_batch(
    next(iter(build_loader("train", BATCH_SIZE))),
    OVERFIT_BATCH_SIZE,
)
overfit_model = MDEN(SMOKE_MODEL_CONFIG)
overfit_result = overfit_one_batch(
    overfit_model,
    fixed_batch,
    DEVICE,
    steps=OVERFIT_STEPS,
    learning_rate=OVERFIT_LR,
    target_rmse=OVERFIT_TARGET_RMSE,
)

assert overfit_result.converged, (
    f"SOC RMSE={overfit_result.soc_rmse[-1]:.6f}, "
    f"SOH RMSE={overfit_result.soh_rmse[-1]:.6f}"
)
print("steps:", overfit_result.steps)
print("SOC RMSE:", overfit_result.soc_rmse[-1])
print("SOH RMSE:", overfit_result.soh_rmse[-1])


## Кривая overfit


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(overfit_result.soc_rmse, label="SOC RMSE")
plt.plot(overfit_result.soh_rmse, label="SOH RMSE")
plt.axhline(
    OVERFIT_TARGET_RMSE,
    color="black",
    linestyle="--",
    label="target",
)
plt.xlabel("Шаг")
plt.ylabel("RMSE")
plt.yscale("log")
plt.grid(alpha=0.3)
plt.legend()
plt.show()
